# クラスの偏ったデータを扱う (クラス重み・しきい値調整)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_classification/imbalanced.ipynb)

NLP クックブック 09-10 ／ テキスト分類

陽性が数 % しかないデータ (スパム、有害投稿、異常報告など) で、少数クラスをきちんと検出したい。

In [ ]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, precision_recall_curve
from sklearn.model_selection import train_test_split

data = fetch_20newsgroups(subset="all", remove=("headers", "footers", "quotes"))
y = (data.target == data.target_names.index("sci.med")).astype(int)   # 「医療」だけを陽性 (約 5%) とする 2 値分類
X_train, X_test, y_train, y_test = train_test_split(data.data, y, test_size=0.3, stratify=y, random_state=0)
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2)
Xtr, Xte = vec.fit_transform(X_train), vec.transform(X_test)
print(f"陽性の割合: {y.mean():.1%}")

for cw in [None, "balanced"]:
    clf = LogisticRegression(C=1, max_iter=1000, class_weight=cw).fit(Xtr, y_train)
    print(f"\nclass_weight={cw}")
    print(classification_report(y_test, clf.predict(Xte), digits=3, target_names=["other", "med"]))

In [ ]:
# しきい値を動かして適合率と再現率のバランスを選ぶ
import matplotlib.pyplot as plt

clf = LogisticRegression(C=1, max_iter=1000).fit(Xtr, y_train)
prob = clf.predict_proba(Xte)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, prob)
f1 = 2 * precision * recall / (precision + recall + 1e-9)
best = f1[:-1].argmax()
print(f"平均適合率 (AP) = {average_precision_score(y_test, prob):.3f}")
print(f"しきい値 0.5 → F1 = {f1[np.searchsorted(thresholds, 0.5)]:.3f} / 最良しきい値 {thresholds[best]:.2f} → F1 = {f1[best]:.3f}")

plt.plot(recall, precision); plt.scatter(recall[best], precision[best], color="red", label="best F1")
plt.xlabel("recall"); plt.ylabel("precision"); plt.legend(); plt.show()

- 不均衡データでは正解率は役に立たない (全部「陰性」と答えても 95%)。少数クラスの適合率・再現率・F1、あるいは PR 曲線下面積 (AP) を見る。
- `class_weight="balanced"` は少数クラスの誤りを重く扱う。再現率が上がり、適合率は下がる。
- しきい値は検証データで決める (テストデータで決めてはいけない)。「見逃しを減らしたい」のか「誤検出を減らしたい」のか、用途に応じて選ぶ。